# Praproses Teks, Representasi TF-IDF, dan Reduksi Dimensi PCA


### Install library terlebih dahulu

In [130]:
!pip install pandas openpyxl numpy scikit-learn

### Import library

In [131]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Membaca dataset

In [132]:
df = pd.read_excel("berita_detik_200_artikel.xlsx")

df.head()

,ID,Isi Berita,Label
0,1,Kejuaraan Nasional Gokart 2026 yang berlangsun...,sport
1,2,PT Bank Negara Indonesia (Persero) Tbk atau BN...,sport
2,3,"Pelatih Timnas voli putra Indonesia, Reidel To...",sport
3,4,Kontingen Indonesia bersiap menatap Asian Game...,sport
4,5,Rizki Juniansyah siap menjalani debut di Asian...,sport


Kode diatas berfungsi untuk membaca file dataset bernama berita_detik_200_artikel.xlsx dan menyimpannya ke dalam variabel df sebagai struktur data tabel (DataFrame) menggunakan library Pandas. Selanjutnya, perintah df.head() digunakan untuk menampilkan 5 baris pertama dari tabel tersebut agar kita dapat memeriksa struktur kolom serta gambaran awal dari isi data berita secara cepat.

### Cek jumlah data:

In [133]:
print("Jumlah data :", len(df))
print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["Label"].value_counts())

Jumlah data : 200

Nama kolom:
Index(['ID', 'Isi Berita', 'Label'], dtype='object')

Jumlah data per label:
Label
sport      100
finance    100
Name: count, dtype: int64


Perintah len(df) menghitung dan menampilkan total baris data yang ada, df.columns menampilkan daftar nama seluruh kolom, sedangkan df["Label"].value_counts() menghitung jumlah kemunculan data berdasarkan masing-masing kategori pada kolom "Label"

### Mengubah label menjadi numerik

In [134]:
df["Label_num"] = df["Label"].map({
    "sport": 1,
    "finance": 0
})

df[["ID", "Label", "Label_num"]].head()

,ID,Label,Label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


Kode diatas bertujuan untuk melakukan enkoding label (mengubah label teks menjadi numerik), yaitu mengubah sport menjadi 1 dan finance menjadi 0 ke dalam kolom baru Label_num. Tampilan 5 baris pertamanya kemudian dipanggil untuk memastikan proses pengubahan angka tersebut berhasil.

In [135]:
print(df["Label_num"].value_counts())

Label_num
1    100
0    100
Name: count, dtype: int64


### Menghitung jumlah kata di semua berita

In [136]:
df["jumlah_kata_asli"] = df["Isi Berita"].astype(str).apply(
    lambda x: len(x.split())
)

df[["ID", "jumlah_kata_asli"]].head()

total_kata = df["jumlah_kata_asli"].sum()

print("Jumlah seluruh kata :", total_kata)

print(df["jumlah_kata_asli"].describe())

Jumlah seluruh kata : 59826
count    200.000000
mean     299.130000
std      164.572732
min       34.000000
25%      201.750000
50%      271.000000
75%      394.500000
max      804.000000
Name: jumlah_kata_asli, dtype: float64


Kode diatas bertujuan untuk menghitung jumlah kata asli pada setiap berita dengan memecah teks pada kolom Isi Berita berdasarkan spasi. Intinya, kode ini membuat kolom baru jumlah_kata_asli, menampilkan lima sampel pertamanya beserta ID, lalu mencetak total seluruh kata dari 200 berita serta ringkasan statistik deskriptifnya (seperti rata-rata, nilai minimum, dan maksimum jumlah kata).

### Melihat jumlah kata setiap berita

In [137]:
df[[
    "ID",
    "Label",
    "jumlah_kata_asli"
]]

,ID,Label,jumlah_kata_asli
0,1,sport,633
1,2,sport,359
2,3,sport,251
3,4,sport,193
4,5,sport,401
...,...,...,...
195,196,finance,378
196,197,finance,386
197,198,finance,59
198,199,finance,321


### Kamus kata tidak baku

In [145]:
kamus_tidak_baku = {
    "tapi": "tetapi",
    "nggak": "tidak",
    "cuma": "hanya",
    "bikin": "membuat",
    "enggak": "tidak",
    "aja": "saja",
    "udah": "sudah",
    "kayak": "seperti",
    "gimana": "bagaimana",
    "gak": "tidak",
    "cuman": "hanya",
    "kayaknya": "sepertinya",
    "nyampe": "sampai",
    "makanya": "oleh karena itu",
    "dr": "dari",
    "ga": "tidak",
    "kalo": "kalau",
    "karna": "karena",
    "trus": "terus",
    "dpt": "dapat",
    "hrs": "harus",
    "sy": "saya",
}

### Kamus bahasa asing

In [146]:
kamus_asing = {
    # --- SPORT ---
    "games": "pertandingan",
    "race": "balapan",
    "rider": "pembalap",
    "racing": "balap",
    "team": "tim",
    "championship": "kejuaraan",
    "runner-up": "juara dua",
    "game": "pertandingan",
    "training": "latihan",
    "match": "pertandingan",
    "manager": "manajer",
    "event": "acara",
    "venue": "lokasi",
    "stage": "tahap",
    "round": "putaran",
    "coach": "pelatih",
    "champion": "juara",
    "league": "liga",

    # --- FINANCE ---
    "financial": "keuangan",
    "investor": "investor",
    "business": "bisnis",
    "sale": "penjualan",
    "market": "pasar",
    "corporate": "korporat",
    "director": "direktur",
    "performance": "kinerja",
    "finance": "keuangan",
    "heritage": "warisan budaya",
    "economy": "ekonomi",
    "loan": "pinjaman",
    "asset": "aset",
    "company": "perusahaan",
    "banking": "perbankan",
}

In [147]:
stopword_list = set([
    "yang", "di", "ke", "dari", "untuk", "pada", "dalam", "dan", "atau", "ini",
    "itu", "juga", "akan", "tersebut", "bisa", "ada", "karena", "oleh", "saat",
    "menjadi", "dengan", "serta", "hingga", "bahwa", "secara", "adalah", "sebagai",
    "ia", "mereka", "kita", "kami", "saya", "anda", "telah", "sudah", "tentang",
    "dapat", "harus", "lain", "hanya", "antara", "para",
    "bagi", "lebih", "melalui", "berbagai", "bersama", "terhadap", "seperti",
    "kembali", "tetapi", "kemudian", "namun", "selain", "lalu", "agar", "sesuai"
])

### Case Folding & Cleaning (URL, Email, Angka, Simbol)

In [141]:
def clean_text_step(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Hapus URL
    text = re.sub(r'\S+@\S+', ' ', text)                # Hapus Email
    text = re.sub(r'\d+', ' ', text)                    # Hapus Angka
    text = re.sub(r'[^a-zA-ZÀ-ÿ\s]', ' ', text)         # Hapus Simbol/Emote
    text = re.sub(r'\s+', ' ', text).strip()            # Hapus Spasi Berlebih
    return text

df['step1_clean'] = df['Isi Berita'].apply(clean_text_step)
df[['Isi Berita', 'step1_clean']].head(2)

,Isi Berita,step1_clean
0,Kejuaraan Nasional Gokart 2026 yang berlangsun...,kejuaraan nasional gokart yang berlangsung dal...
1,PT Bank Negara Indonesia (Persero) Tbk atau BN...,pt bank negara indonesia persero tbk atau bni ...


Fungsi clean_text_step ini bertugas untuk membersihkan teks berita pada tahap awal (cleaning) dengan mengubah semua huruf menjadi kecil (lowercasing), serta menghapus URL, email, angka, dan semua simbol atau emotikon menggunakan regex (regular expression). Selain itu, fungsi ini merapikan spasi berlebih sebelum diterapkan secara otomatis ke setiap baris pada kolom "Isi Berita" dan hasilnya disimpan ke dalam kolom baru "step1_clean". Kode diakhiri dengan perintah df[['Isi Berita', 'step1_clean']].head(2) untuk menampilkan perbandingan dua baris pertama antara teks berita asli dengan teks yang sudah dibersihkan.

### Normalisasi (Kata Baku & Bahasa Asing)

In [142]:
def normalize_step(text):
    kata = text.split()
    hasil = []
    for token in kata:
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]
        if token in kamus_asing:
            token = kamus_asing[token]
        hasil.append(token)
    return " ".join(hasil)

df['step2_normalized'] = df['step1_clean'].apply(normalize_step)
df[['step1_clean', 'step2_normalized']].head(2)

,step1_clean,step2_normalized
0,kejuaraan nasional gokart yang berlangsung dal...,kejuaraan nasional gokart yang berlangsung dal...
1,pt bank negara indonesia persero tbk atau bni ...,pt bank negara indonesia persero tbk atau bni ...


Fungsi normalize_step ini bertugas untuk melakukan normalisasi kata dengan memecah teks menjadi daftar kata (token), lalu memeriksa setiap kata tersebut ke dalam kamus kata tidak baku dan kamus bahasa asing. Jika kata ditemukan di kamus_tidak_baku, maka kata tersebut diganti menjadi kata baku, dan jika ada di kamus_asing, kata tersebut diterjemahkan ke dalam bahasa Indonesia. Setelah semua kata dinormalisasi, teks digabungkan kembali menjadi satu kalimat utuh dan hasilnya disimpan ke dalam kolom baru bernama "step2_normalized" melalui proses pemetaan pada kolom "step1_clean". Kode diakhiri dengan df[['step1_clean', 'step2_normalized']].head(2) untuk menampilkan perbandingan dua baris pertama antara teks sebelum dan sesudah tahap normalisasi.

### Stopword Removal (Hasil Akhir)

In [143]:
def stopword_step(text):
    kata = text.split()
    hasil = [token for token in kata if token not in stopword_list and len(token) > 1]
    return " ".join(hasil)

df['berita_clean'] = df['step2_normalized'].apply(stopword_step)
df[['step2_normalized', 'berita_clean']].head(2)

,step2_normalized,berita_clean
0,kejuaraan nasional gokart yang berlangsung dal...,kejuaraan nasional gokart berlangsung enam put...
1,pt bank negara indonesia persero tbk atau bni ...,pt bank negara indonesia persero tbk bni membe...


Fungsi stopword_step ini bertujuan untuk menghapus kata-kata umum atau kata penghubung (stopword) serta kata yang hanya terdiri dari satu karakter dari dalam teks. Teks dari kolom "step2_normalized" dipecah menjadi daftar kata, lalu disaring sesuai aturan tersebut, dan digabungkan kembali menjadi satu kalimat utuh pada kolom baru bernama "berita_clean". Perintah df[['step2_normalized', 'berita_clean']].head(2) di akhir kode digunakan untuk membandingkan cuplikan dua baris pertama antara teks sebelum dan sesudah tahap stopword removal.

### Menerapkan preprocessing ke semua berita

In [150]:
df["berita_clean"] = df["Isi Berita"].apply(preprocessing)

df[[
    "ID",
    "Isi Berita",
    "berita_clean"
]].head()

,ID,Isi Berita,berita_clean
0,1,Kejuaraan Nasional Gokart 2026 yang berlangsun...,kejuaraan nasional gokart berlangsung enam put...
1,2,PT Bank Negara Indonesia (Persero) Tbk atau BN...,pt bank negara indonesia persero tbk bni membe...
2,3,"Pelatih Timnas voli putra Indonesia, Reidel To...",pelatih timnas voli putra indonesia reidel toi...
3,4,Kontingen Indonesia bersiap menatap Asian Game...,kontingen indonesia bersiap menatap asian pert...
4,5,Rizki Juniansyah siap menjalani debut di Asian...,rizki juniansyah siap menjalani debut asian pe...


### Hitung jumlah kata setelah preprocessing

In [151]:
df["jumlah_kata_clean"] = df["berita_clean"].apply(
    lambda x: len(x.split())
)

df[[
    "ID",
    "jumlah_kata_asli",
    "jumlah_kata_clean"
]].head()

,ID,jumlah_kata_asli,jumlah_kata_clean
0,1,633,468
1,2,359,272
2,3,251,189
3,4,193,144
4,5,401,284


Kode tersebut bertujuan untuk menghitung jumlah kata setelah pembersihan (preprocessing) pada setiap berita dengan memecah teks pada kolom "berita_clean" berdasarkan spasi. Intinya, kode ini membuat kolom baru bernama "jumlah_kata_clean", lalu menampilkan lima baris pertama dari kolom "ID", "jumlah_kata_asli", dan "jumlah_kata_clean" untuk membandingkan jumlah kata sebelum dan sesudah proses preprocessing.

In [152]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 59826
Total kata setelah preprocessing : 44469


### Ekstrak seluruh kata unik

In [153]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print(
    "Jumlah seluruh kata:",
    len(semua_kata)
)

print(
    "Jumlah kata unik:",
    len(kata_unik)
)

Jumlah seluruh kata: 44469
Jumlah kata unik: 6905


Kode tersebut bertujuan untuk mengekstrak seluruh kata dan menemukan daftar kata unik dari seluruh berita yang telah dibersihkan pada kolom "berita_clean". Intinya, kode ini mengumpulkan seluruh kata dari setiap berita ke dalam satu daftar, menghapus kata-kata yang duplikat dengan set(), mengurutkannya secara alfabetis dengan sorted(), lalu mencetak total keseluruhan kata beserta total kata unik yang ditemukan.

In [154]:
kata_unik[:100]

['aaron',
 'abadi',
 'abdul',
 'abdullah',
 'abimanyu',
 'abraham',
 'absen',
 'absennya',
 'absorber',
 'abu',
 'ac',
 'acara',
 'acd',
 'aceh',
 'ach',
 'achmad',
 'acosta',
 'across',
 'activities',
 'activity',
 'adam',
 'adanya',
 'adaptasi',
 'adaptif',
 'adapun',
 'adella',
 'adem',
 'adi',
 'adil',
 'adininggar',
 'adisti',
 'adisutjipto',
 'adk',
 'administrasi',
 'adopsi',
 'adp',
 'adrian',
 'adu',
 'adventure',
 'aerodinamika',
 'aerodrome',
 'aeronautika',
 'af',
 'aff',
 'affected',
 'afriani',
 'agak',
 'agam',
 'agama',
 'agenda',
 'agent',
 'agoeng',
 'agresif',
 'agrinas',
 'agro',
 'agunan',
 'agung',
 'agus',
 'agustus',
 'ah',
 'ahhn',
 'ahi',
 'ahli',
 'ahlinya',
 'ahmad',
 'ahy',
 'ai',
 'aichi',
 'aida',
 'air',
 'airbus',
 'airin',
 'airlangga',
 'airline',
 'airlines',
 'airnav',
 'airport',
 'ajaknya',
 'ajang',
 'ajukan',
 'akademisi',
 'akar',
 'akbar',
 'akd',
 'akhir',
 'akhirnya',
 'akibat',
 'akibatnya',
 'akn',
 'akomodasi',
 'akrab',
 'akses',
 'akses

In [155]:
df_kata_unik = pd.DataFrame({
    "kata_unik": kata_unik
})

df_kata_unik.head(20)

,kata_unik
0,aaron
1,abadi
2,abdul
3,abdullah
4,abimanyu
5,abraham
6,absen
7,absennya
8,absorber
9,abu


In [156]:
df_kata_unik.to_excel(
    "kata_unik.xlsx",
    index=False
)

### Membagi data 160 training dan 40 testing

In [161]:
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df["berita_clean"],
    df["Label_num"],

    test_size=40,

    random_state=42,

    stratify=df["Label_num"]
)

print(
    "Jumlah training:",
    len(X_train_text)
)

print(
    "Jumlah testing:",
    len(X_test_text)
)

Jumlah training: 160
Jumlah testing: 40


Kode tersebut bertujuan untuk membagi dataset menjadi data latih (training) dan data uji (testing) menggunakan fungsi train_test_split. Intinya, kode ini memisahkan teks berita (df["berita_clean"]) dan label kategorinya (df["Label_num"]) dengan mengalokasikan sebanyak 40 data sebagai data uji (test_size=40) serta 160 sisanya sebagai data latih. Penggunaan stratify=df["Label_num"] memastikan proporsi antar-label pada data latih dan uji tetap seimbang, kemudian kode diakhiri dengan mencetak jumlah masing-masing kumpulan data tersebut untuk verifikasi.

In [163]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
Label_num
0    80
1    80
Name: count, dtype: int64

TESTING
Label_num
0    20
1    20
Name: count, dtype: int64


### Representasi TF-IDF

In [164]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

In [165]:
nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 2849


Kode tersebut bertujuan untuk mengubah data teks menjadi matriks angka (bobot TF-IDF) dengan membuang kata yang muncul di kurang dari 2 berita (min_df=2) serta kata yang terlalu sering muncul di atas 95% berita (max_df=0.95). Intinya, kode ini mempelajari pola kosa kata sekaligus mentransformasi data latih (X_train_text) menggunakan fit_transform(), mentransformasi data uji (X_test_text) menggunakan transform(), lalu mencetak total jumlah kata atau fitur unik yang berhasil diekstrak.

### Melihat matriks TF-IDF

In [166]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,abimanyu,absen,abu,acara,acd,acosta,adanya,adapun,adi,administrasi,...,yuk,yuki,yusuf,zaki,zandvoort,zarco,zero,zona,zulhas,zulkifli
0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,0.0,0.0,0.000000,0.186011,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.041141
3,0.0,0.0,0.031585,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.04977,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


Kode tersebut bertujuan untuk mengubah matriks bobot TF-IDF data latih menjadi tabel (DataFrame) yang rapi dan mudah dibaca. Intinya, kode ini mengonversi matriks TF-IDF (X_train_tfidf) dari format terkompresi (sparse matrix) menjadi array biasa, menetapkan nama kata-kata (nama_fitur) sebagai judul kolom, lalu menampilkan lima baris pertamanya melalui perintah tfidf_train_df.head() untuk melihat nilai bobot setiap kata pada sampel berita data latih.

In [167]:
tfidf_train_df["Label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.head()

,abimanyu,absen,abu,acara,acd,acosta,adanya,adapun,adi,administrasi,...,yuki,yusuf,zaki,zandvoort,zarco,zero,zona,zulhas,zulkifli,Label
0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0
1,0.0,0.0,0.000000,0.186011,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.041141,0
3,0.0,0.0,0.031585,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1
4,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.04977,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1


Kode tersebut bertujuan untuk menggabungkan label target ke dalam tabel matriks TF-IDF data latih. Intinya, kode ini mereset indeks dari variabel y_train agar sesuai dengan urutan baris tabel, memasukkannya ke dalam kolom baru bernama "Label", lalu menampilkan lima baris pertama dari tfidf_train_df untuk memastikan label kategori berita sudah terpasang dengan benar di samping nilai bobot TF-IDF.

### Menyimpan data TF-IDF

In [168]:
tfidf_train_df.to_excel(
    "tfidf_training.xlsx",
    index=False
)

In [169]:
tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["Label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "tfidf_testing.xlsx",
    index=False
)

In [170]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

In [171]:
for kelas in sorted(
    y_train.unique()
):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]


    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )


    jumlah_dokumen_class = (
        X_class.shape[0]
    )


    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )


    class_space_density += (
        class_density
    )

In [172]:
epsilon = 1e-12

icsdf = np.log(

    (jumlah_kelas + epsilon)

    /

    (class_space_density + epsilon)
)

In [173]:
df_icsdf = pd.DataFrame({

    "kata": features,

    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=Falsea
).head(20)

,kata,ICSDF
13,agam,4.382027
2831,yansyah,4.382027
2830,yani,4.382027
1652,menyusun,4.382027
1655,meramaikan,4.382027
1656,merambah,4.382027
1661,merchandise,4.382027
1665,merespons,4.382027
1670,mesti,4.382027
1671,meter,4.382027


Kode tersebut bertujuan untuk menampilkan 20 kata dengan bobot diskriminatif tertinggi berdasarkan hasil perhitungan Inter-Class Space Density Frequency (ICSDF). Intinya, kode ini membuat tabel (DataFrame) baru yang memuat daftar kata (features) beserta skor ICSDF-nya, mengurutkan nilainya dari yang paling besar ke paling kecil (descending), lalu mengambil 20 kata teratas melalui fungsi .head(20) untuk melihat kata mana saja yang paling kuat dalam membedakan antar-kategori berita.

### TF-IDF × ICSDF

In [174]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

In [175]:
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

Kode tersebut bertujuan untuk mengalikan bobot TF-IDF dengan nilai bobot ICSDF (Inter-Class Space Density Frequency) pada data latih dan data uji. Intinya, kode ini melakukan pembobotan ulang agar kata-kata yang paling penting dan kontras antarkategori mendapat skor lebih tinggi, lalu mengonversi kembali hasil perkalian tersebut ke dalam format matriks terkompresi (csr_matrix) agar efisien dari segi memori dan siap untuk proses seleksi fitur selanjutnya.

## Menentukan kata yang paling penting

In [176]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

In [177]:
TOP_K = 100

In [178]:
top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

In [179]:
kata_penting = features[
    top_index
]

kata_penting[:30]

array(['sep', 'detikupdate', 'marquez', 'motogp', 'pertandingan', 'asian',
       'purbaya', 'aragon', 'beras', 'marc', 'bandara', 'balapan',
       'anggaran', 'detik', 'rp', 'wib', 'video', 'triliun', 'posisi',
       'ekonomi', 'tim', 'gunakan', 'tombol', 'panah', 'klik', 'run',
       'bezzecchi', 'ihr', 'pemain', 'menjelajahi'], dtype=object)

In [180]:
df_kata_penting = pd.DataFrame({

    "kata": kata_penting,

    "skor": skor_fitur[
        top_index
    ]
})

df_kata_penting.head(30)

,kata,skor
0,sep,0.078293
1,detikupdate,0.073465
2,marquez,0.071191
3,motogp,0.055552
4,pertandingan,0.051298
5,asian,0.047652
6,purbaya,0.044324
7,aragon,0.044151
8,beras,0.043345
9,marc,0.040718


Kode tersebut bertujuan untuk menampilkan daftar kata-kata paling penting beserta nilai skor kontribusinya. Intinya, kode ini membuat tabel (DataFrame) bernama df_kata_penting yang menyatukan nama-nama kata kunci terpilih (kata_penting) dengan nilai skor pembobotannya (skor_fitur), lalu menampilkan 30 kata teratas dengan skor tertinggi menggunakan perintah df_kata_penting.head(30) untuk mempermudah analisis data.

### Reduksi menjadi 100 fitur ICSDF

In [185]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 2849)
Sesudah ICSDF: (160, 100)


Kode tersebut bertujuan untuk melakukan seleksi fitur dengan mengambil kolom kata-kata terbaik hasil pembobotan ICSDF. Intinya, kode ini memfilter matriks data latih (X_train_icsdf) dan data uji (X_test_icsdf) agar hanya menyisakan kata-kata paling penting berdasarkan urutan indeks top_index, lalu mencetak perbandingan ukuran dimensi matriks sebelum dan sesudah proses seleksi fitur untuk memastikan jumlah kata berhasil dikurangi.

### Tabel hasil ICSDF

In [187]:
icsdf_train_df = pd.DataFrame(

    X_train_selected.toarray(),

    columns=kata_penting
)

icsdf_train_df["Label"] = (

    y_train.reset_index(
        drop=True
    )
)

icsdf_train_df.head()

,sep,detikupdate,marquez,motogp,pertandingan,asian,purbaya,aragon,beras,marc,...,finis,aprilia,motor,daniel,energi,gubernur,kai,masing,kelas,Label
0,1.367161,1.676439,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0
1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,1.244335,0.0,0.0,0.0,0.0,0.200267,0.000000,1
2,1.370088,1.680028,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.039948,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.204597,1
4,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.144321,0.0,0.0,...,0.129715,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,1


In [188]:
icsdf_train_df.to_excel(
    "icsdf_training.xlsx",
    index=False
)

Kode tersebut bertujuan untuk menyajikan hasil seleksi fitur ke dalam bentuk tabel (DataFrame) terstruktur beserta labelnya. Intinya, kode ini mengubah matriks data latih yang sudah terseleksi (X_train_selected) menjadi tabel dengan nama-nama kata penting sebagai judul kolomnya, menyatukannya dengan kolom "Label" dari data latih (y_train), lalu menampilkan lima baris pertama tabel tersebut untuk mengecek hasil pembobotan kata-kata terbaik.

### PCA

In [189]:
pca = PCA(

    n_components=20,

    random_state=42
)

In [190]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

In [191]:
X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

In [192]:
print(
    X_train_pca.shape
)

print(
    X_test_pca.shape
)

(160, 20)
(40, 20)


Kode tersebut bertujuan untuk memeriksa dimensi matriks data hasil reduksi dimensi menggunakan PCA (Principal Component Analysis). Intinya, perintah ini mencetak jumlah baris (sampel berita) dan jumlah kolom (komponen utama/fitur baru) pada data latih (X_train_pca) serta data uji (X_test_pca) untuk memastikan proses reduksi dimensi telah berhasil mengurangi jumlah fitur sesuai target.

### Membuat tabel data reduksi training

In [193]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

In [194]:
df_train_reduksi = pd.DataFrame(

    X_train_pca,

    columns=nama_pc
)

df_train_reduksi["Label"] = (

    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,Label
0,2.132895,0.108502,-0.172438,-0.019958,-0.034626,-0.010389,0.142491,-0.072439,-0.014625,-0.016768,...,-0.023355,0.006694,0.003800,-0.001793,-0.004379,0.000022,-0.008442,-0.017503,-0.009090,0
1,-0.103432,-0.030254,-0.052969,-0.001502,-0.047263,0.028127,0.007489,0.097061,-0.078200,-0.098604,...,0.077766,-0.035842,0.115162,0.000122,0.021443,-0.115786,-0.029852,-0.000907,-0.192764,1
2,2.135697,0.111018,-0.171703,-0.015911,-0.034005,-0.011117,0.139932,-0.072336,-0.014495,-0.015709,...,-0.023567,0.006832,0.001562,-0.002214,-0.005644,0.004346,-0.018062,-0.025027,-0.000920,0
3,-0.113418,0.046309,-0.097065,0.013378,-0.077760,0.053639,0.024920,0.066693,-0.071916,-0.002167,...,0.069315,-0.062517,0.001796,0.186749,-0.019185,0.002176,-0.003519,0.013036,-0.073658,1
4,-0.141491,0.291807,-0.119716,-0.004239,-0.110929,0.052760,0.093543,0.182452,-0.205433,0.062999,...,1.102597,-0.755134,-0.374562,-0.294553,-0.006868,0.051245,-0.003760,-0.052318,0.065476,1


Kode tersebut bertujuan untuk mengubah hasil reduksi dimensi PCA data latih menjadi tabel (DataFrame) terstruktur yang dilengkapi label. Intinya, kode ini mengonversi matriks X_train_pca menjadi tabel dengan kolom berlabel nama komponen utama (nama_pc), menggabungkannya dengan kolom "Label" dari y_train, lalu menampilkan lima baris pertama tabel tersebut untuk melihat gambaran data yang siap diekspor ke Orange Data Mining.

### Data testing

In [195]:
df_test_reduksi = pd.DataFrame(

    X_test_pca,

    columns=nama_pc
)

df_test_reduksi["Label"] = (

    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,Label
0,-0.078981,-0.093776,-0.035948,-0.029933,-0.047618,0.007547,0.022616,0.023932,-0.056846,-0.036976,...,0.004079,-0.008926,0.025012,0.017782,0.022994,-0.046430,0.154690,0.104674,-0.036420,0
1,-0.082630,-0.058729,-0.080313,0.026288,-0.074101,-0.022530,-0.027983,0.048638,-0.028524,-0.017459,...,-0.003367,-0.021312,-0.024743,0.011974,0.121570,0.013817,-0.001841,0.014139,-0.036705,1
2,-0.104655,-0.071327,-0.060899,-0.007222,-0.073223,0.039303,-0.014174,0.166647,-0.069683,-0.106793,...,0.095647,-0.039146,0.291446,-0.065199,-0.005112,-0.123784,-0.055499,-0.096319,0.174229,1
3,-0.109570,-0.023019,-0.126472,0.047342,-0.083203,-0.080200,-0.067861,-0.017497,0.027699,-0.028928,...,-0.050599,0.002943,-0.022416,0.012929,-0.016727,-0.019746,-0.017609,0.027132,-0.136827,1
4,-0.087235,-0.140852,0.037065,-0.098723,0.062720,0.016566,-0.007813,0.007058,-0.063811,-0.027025,...,-0.003922,0.000268,0.001973,0.008838,-0.000142,0.006508,-0.087469,0.002535,-0.111467,0


Kode tersebut bertujuan untuk mengubah hasil reduksi dimensi PCA pada data uji menjadi tabel (DataFrame) terstruktur yang dilengkapi label. Intinya, kode ini mengonversi matriks X_test_pca menjadi tabel dengan nama komponen utama (nama_pc) sebagai kolomnya, menyatukannya dengan kolom "Label" dari data uji (y_test), lalu menampilkan lima baris pertama tabel tersebut untuk melihat gambaran data uji yang siap diproses lebih lanjut.

### Menyimpan data reduksi

In [196]:
df_train_reduksi.to_excel(
    "reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "reduksi_testing.xlsx",
    index=False
)

### Cek total training dan testing

In [197]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


Kode tersebut bertujuan untuk memeriksa dimensi akhir dari tabel data latih dan data uji hasil reduksi PCA. Intinya, perintah ini mencetak total baris (sampel berita) serta total kolom (komponen PCA ditambah kolom label) pada df_train_reduksi dan df_test_reduksi untuk memastikan struktur kedua tabel tersebut sudah sesuai sebelum diolah atau diekspor.

### Kolom terakhir harus label

In [198]:
df_train_reduksi.columns

Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19',
       'PC20', 'Label'],
      dtype='object')